# 01 · Adquisición de datos

**Qué hace este notebook:** lee cada fuente, la describe y la deja en `data/interim/` en un formato cómodo (parquet) y con un esquema unificado.

**Contexto.** El objetivo del proyecto es predecir la demanda de bicicletas por estación y franja horaria. En la corrección de la propuesta, la cátedra marcó que hay que tener en cuenta *las bicis rotas que ocupan lugar y no se pueden usar*. Por eso, además de los viajes, se incorporan fotos periódicas del estado de cada estación, que informan cuántas bicis y anclajes están fuera de servicio.

## 0. Configuración

- **`DATA_DIR`:** carpeta con los datos crudos. Por defecto es `datasets/` en la raíz del repositorio (se puede cambiar con la variable de entorno `ECOBICI_DATA_DIR`). Ver el README para descargarlos. Estructura esperada:

  ```
  datasets/
  ├── buenos-aires.zip            # fotos GBFS y pronósticos (o ya descomprimido como archive/)
  └── ba-ecobici-datasets/
      ├── recorridos-realizados-2024.zip
      ├── recorridos-realizados-2025.zip
      ├── recorridos-realizados-2026.zip
      └── ciclovias.csv
  ```

- **Descompresión automática:** la celda de configuración descomprime `buenos-aires.zip` y los ZIP de recorridos la primera vez que se corre (los CSV de viajes ocupan ~2 GB descomprimidos). Las corridas siguientes reutilizan lo ya extraído.
- **DuckDB:** los archivos grandes (21 M de filas de fotos y ~2 GB de viajes) se leen y agregan con DuckDB, que ejecuta SQL directamente sobre los archivos sin cargarlos enteros en memoria. Los resultados, ya chicos, se pasan a pandas.


In [1]:
%pip install -q -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import os
import shutil
import time
import zipfile

import duckdb
import pandas as pd
import requests

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

REPO_DIR = Path("..").resolve()

DATA_DIR = Path(os.environ.get("ECOBICI_DATA_DIR", REPO_DIR / "datasets")).resolve()

FOTOS_DIR = DATA_DIR / "archive" / "stations" / "buenos-aires" / "ecobici"
PRONOSTICOS_DIR = DATA_DIR / "archive" / "weather" / "buenos-aires"
BA_DATA_DIR = DATA_DIR / "ba-ecobici-datasets"

INTERIM_DIR = REPO_DIR / "data" / "interim"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

TZ = "America/Argentina/Buenos_Aires"
ANIOS = [2024, 2025, 2026]

assert DATA_DIR.exists(), f"No se encontró {DATA_DIR}. Descargar los datos como indica el README (o definir ECOBICI_DATA_DIR)."

# Fotos y pronósticos: vienen dentro de buenos-aires.zip (carpeta archive/)
if not FOTOS_DIR.exists():
    zip_fotos = DATA_DIR / "buenos-aires.zip"
    assert zip_fotos.exists(), f"No se encontró {FOTOS_DIR} ni {zip_fotos}. Ver el README."
    print("Descomprimiendo", zip_fotos.name, "...")
    with zipfile.ZipFile(zip_fotos) as z:
        z.extractall(DATA_DIR / "_buenos-aires")
    # El zip puede traer la carpeta archive/ o directamente stations/ y weather/: se busca y se mueve a DATA_DIR/archive
    raiz = next(p.parent for p in (DATA_DIR / "_buenos-aires").rglob("stations")
                if "__MACOSX" not in p.parts and (p / "buenos-aires" / "ecobici").exists())
    raiz.rename(DATA_DIR / "archive")
    shutil.rmtree(DATA_DIR / "_buenos-aires", ignore_errors=True)

# Viajes: un ZIP por año con un único CSV adentro; se extrae a recorridos-realizados-AAAA/
for anio in ANIOS:
    destino = BA_DATA_DIR / f"recorridos-realizados-{anio}"
    if not any(destino.glob("*.csv")):
        zip_viajes = BA_DATA_DIR / f"recorridos-realizados-{anio}.zip"
        assert zip_viajes.exists(), f"No se encontró {zip_viajes}. Ver el README."
        print("Descomprimiendo", zip_viajes.name, "...")
        with zipfile.ZipFile(zip_viajes) as z:
            z.extractall(destino)

for carpeta in [FOTOS_DIR, PRONOSTICOS_DIR, BA_DATA_DIR]:
    assert carpeta.exists(), f"No se encontró {carpeta}. Revisar la estructura de datos del README."

print("Datos crudos:        ", DATA_DIR)
print("Salidas intermedias: ", INTERIM_DIR)


Datos crudos:         /Users/juan.fernandez/itba/cda/ecobici-demand-predictor/datasets
Salidas intermedias:  /Users/juan.fernandez/itba/cda/ecobici-demand-predictor/.claude/worktrees/code-review-consigna-notebooks-9f6cbd/data/interim


In [3]:
con = duckdb.connect()
con.execute("SET memory_limit = '4GB'")    # límite para que funcione en máquinas con poca RAM
con.execute(f"SET TimeZone = '{TZ}'")      # las fechas con zona horaria se muestran en hora de Buenos Aires


def consulta(sql):
    """Ejecuta una consulta SQL en DuckDB y devuelve el resultado como DataFrame de pandas."""
    return con.sql(sql).df()


print("DuckDB", duckdb.__version__, "| pandas", pd.__version__)

DuckDB 1.5.6 | pandas 2.3.3


## 1. Fuentes de datos: origen y formato

| # | Fuente | Origen | Formato | Cómo se obtiene | Rol en el proyecto |
|---|---|---|---|---|---|
| 1 | Fotos del estado de estaciones (GBFS) | Repositorio `MaxHalford/bike-sharing-history` | Parquet, un archivo por mes | Descarga de los archivos mensuales | Disponibilidad, **bicis y anclajes rotos**, estaciones críticas |
| 2 | Viajes realizados 2024, 2025 y 2026 | BA Data (GCBA), dataset "Bicicletas públicas" | CSV, un archivo por año | Descarga manual del portal | **Target**: retiros y arribos por estación y hora |
| 3 | Clima observado | Open-Meteo, Historical Weather API | JSON (API REST) | `requests.get` | Temperatura, precipitación y viento por hora |
| 4 | Feriados nacionales | ArgentinaDatos | JSON (API REST) | `requests.get` | Variables de calendario |
| 5 | Pronósticos de clima | Repositorio `MaxHalford/bike-sharing-history` | Parquet, un archivo por mes | Descarga de los archivos mensuales | Sólo como referencia: en producción sólo habrá pronóstico |
| 6 | Ciclovías (opcional) | BA Data | CSV con geometría WKT | Descarga manual del portal | Posible variable espacial |

Las fuentes 2 y 6 se bajaron a mano porque el portal de BA Data bloquea las descargas automáticas. Las fuentes 3 y 4 se consultan en vivo desde este notebook.

## 2. Fotos del estado de estaciones (GBFS)

**Qué son.** El sistema de Ecobici publica en tiempo real, con el estándar abierto GBFS, el estado de cada estación: cuántas bicis hay disponibles, cuántas están rotas, cuántos "slots" están libres, etc. El repositorio `bike-sharing-history` guarda una "foto" de ese estado cada ~15 minutos con la técnica de *git scraping* (cada foto es un commit) y la publica en archivos parquet mensuales.

**Por qué la usamos.** Es la única fuente con las bicis rotas (`num_bikes_disabled`) y los "slots" fuera de servicio (`num_docks_disabled`), que es lo que marcó la cátedra en la corrección de la primer entrega (la propuesta).

**Variables:**

| Variable | Tipo | Significado |
|---|---|---|
| `city`, `provider` | texto | Ciudad (`buenos-aires`) y operador (`ecobici`) |
| `station_id` | texto | Identificador de la estación. **Es la clave para cruzar con los viajes.** No coincide con el número que aparece en el nombre |
| `name` | texto | Nombre de la estación, con el formato `"NNN - NOMBRE"` |
| `short_name`, `external_id` | texto | Identificadores alternativos (casi siempre vacíos) |
| `longitude`, `latitude` | decimal | Ubicación (WGS84) |
| `capacity` | entero | Capacidad publicada (anclajes totales) |
| `num_bikes_available` | entero | Bicis disponibles para alquilar |
| `num_bikes_disabled` | entero | **Bicis rotas o bloqueadas**, que ocupan un anclaje |
| `num_docks_available` | entero | Anclajes libres para devolver una bici |
| `num_docks_disabled` | entero | Anclajes fuera de servicio |
| `is_installed`, `is_renting`, `is_returning` | booleano | Si la estación está instalada, si permite alquilar y si permite devolver |
| `status` | texto | Estado declarado (`IN_SERVICE`, `MAINTENANCE`) |
| `committed_at_utc` | fecha y hora (UTC) | Momento de la foto |
| `git_commit`, `source_path` | texto | Trazabilidad: commit y archivo de origen en el repositorio |

In [4]:
archivos_fotos = sorted(FOTOS_DIR.glob("*/*.parquet"))
print(f"{len(archivos_fotos)} archivos, {sum(a.stat().st_size for a in archivos_fotos) / 1e6:.1f} MB en total")
pd.DataFrame({
    "archivo": [a.relative_to(FOTOS_DIR).as_posix() for a in archivos_fotos],
    "MB": [round(a.stat().st_size / 1e6, 1) for a in archivos_fotos],
})

27 archivos, 36.1 MB en total


,archivo,MB
0,2024/Apr.parquet,0.1
1,2024/Aug.parquet,0.5
2,2024/Dec.parquet,1.6
3,2024/Nov.parquet,1.6
4,2024/Oct.parquet,1.9
5,2024/Sep.parquet,1.8
6,2025/Apr.parquet,1.9
7,2025/Aug.parquet,1.5
8,2025/Dec.parquet,1.7
9,2025/Feb.parquet,1.8


Se crea una *vista* de DuckDB sobre todos los archivos: se consultan como si fueran una sola tabla.

In [5]:
con.execute(f"CREATE OR REPLACE VIEW fotos AS SELECT * FROM read_parquet('{FOTOS_DIR.as_posix()}/*/*.parquet')")
consulta("DESCRIBE fotos")[["column_name", "column_type"]]

,column_name,column_type
0,city,VARCHAR
1,provider,VARCHAR
2,station_id,VARCHAR
3,name,VARCHAR
4,short_name,VARCHAR
5,external_id,VARCHAR
6,longitude,DOUBLE
7,latitude,DOUBLE
8,capacity,BIGINT
9,num_bikes_available,BIGINT


In [6]:
consulta("SELECT * FROM fotos USING SAMPLE 5")

,city,provider,station_id,name,short_name,external_id,longitude,latitude,capacity,num_bikes_available,num_bikes_disabled,num_docks_available,num_docks_disabled,is_installed,is_renting,is_returning,status,committed_at_utc,git_commit,source_path
0,buenos-aires,ecobici,242,103 - MALBA,None,None,-58.403984,-34.576949,32,3,0,29,0,True,True,True,IN_SERVICE,2024-04-10 21:48:47-03:00,95ba3371f297fadbb03a9e6b96e48aa03f16c585,data/stations/buenos-aires/ecobici.geojson
1,buenos-aires,ecobici,309,298 - PLAZA DR. JUAN B. TERAN,None,None,-58.519781,-34.621412,16,11,3,2,0,True,True,True,IN_SERVICE,2024-04-11 02:05:19-03:00,46ff27d5a525833e3f8660cd053764729d50ce83,data/stations/buenos-aires/ecobici.geojson
2,buenos-aires,ecobici,457,110 - PLAZA CD DE BANFF,None,None,-58.521740,-34.628050,16,7,2,7,0,True,True,True,IN_SERVICE,2024-04-11 06:21:17-03:00,b5bf29eb1f0715f8a496667587125baede770c69,data/stations/buenos-aires/ecobici.geojson
3,buenos-aires,ecobici,497,307 - CERRETTI,None,None,-58.496232,-34.574424,12,5,3,4,0,True,True,True,IN_SERVICE,2024-04-11 07:34:23-03:00,ace36eb1639fc8094ad8389b04d9a258177e05f0,data/stations/buenos-aires/ecobici.geojson
4,buenos-aires,ecobici,496,052 - CAMPANA II,None,None,-58.513166,-34.583160,12,4,3,5,0,True,True,True,IN_SERVICE,2024-04-11 08:34:09-03:00,8610879253e3cf2d1b00a88687699486d1436d9b,data/stations/buenos-aires/ecobici.geojson


Para ver los tipos de pandas y los no nulos se toma una muestra aleatoria de 100.000 filas (el total de filas no entra cómodo en memoria).

In [7]:
muestra_fotos = consulta("SELECT * FROM fotos USING SAMPLE 100000")
muestra_fotos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 20 columns):
 #   Column               Non-Null Count   Dtype                                         
---  ------               --------------   -----                                         
 0   city                 100000 non-null  object                                        
 1   provider             100000 non-null  object                                        
 2   station_id           100000 non-null  object                                        
 3   name                 100000 non-null  object                                        
 4   short_name           77374 non-null   object                                        
 5   external_id          11251 non-null   object                                        
 6   longitude            100000 non-null  float64                                       
 7   latitude             100000 non-null  float64                              

In [8]:
consulta("""
SELECT count(*)                                                        AS filas,
       count(DISTINCT committed_at_utc)                                AS fotos,
       count(DISTINCT station_id)                                      AS estaciones,
       min(committed_at_utc)                                           AS primera_foto,
       max(committed_at_utc)                                           AS ultima_foto,
       count(*) FILTER (WHERE TRY_CAST(station_id AS INTEGER) IS NULL) AS station_id_no_numerico
FROM fotos
""")

,filas,fotos,estaciones,primera_foto,ultima_foto,station_id_no_numerico
0,21025533,53708,414,2024-04-10 18:45:34-03:00,2026-09-27 11:01:36-03:00,0


Fotos por mes. Un mes completo tiene ~2.700 fotos (una cada ~16 minutos). Los meses con muchas menos muestran huecos en la recolección, que se analizan en el segundo notebook.

In [9]:
fotos_por_mes = consulta(f"""
SELECT strftime(committed_at_utc AT TIME ZONE '{TZ}', '%Y-%m') AS mes,
       count(DISTINCT committed_at_utc)                         AS fotos,
       count(*)                                                 AS filas
FROM fotos
GROUP BY 1
ORDER BY 1
""")
fotos_por_mes

,mes,fotos,filas
0,2024-04,158,55461
1,2024-08,856,314921
2,2024-09,2691,992300
3,2024-10,2765,1026429
4,2024-11,2655,1008491
5,2024-12,2737,1056482
6,2025-01,2759,1087118
7,2025-02,2495,992232
8,2025-03,2706,1079231
9,2025-04,2569,1024868


### 2.1 Agregación por estación y hora

El proyecto trabaja por estación y hora, así que las fotos se resumen en una fila por estación y hora. Decisiones:

- **Hora local.** Las fotos vienen en UTC. Se pasan a hora de Buenos Aires (`AT TIME ZONE`) antes de truncar a la hora, porque los viajes vienen en hora local.
- **`station_id` como entero,** igual que en los viajes, para poder cruzarlos después.
- **Promedios dentro de la hora** para las cantidades (disponibles, rotas, docks).
- **Capacidad útil** = bicis disponibles + docks libres. Es lo que un usuario realmente puede usar: descuenta las bicis rotas que ocupan lugar y los docks fuera de servicio.
- **Proporción del tiempo vacía o llena:** fracción de las fotos de esa hora con 0 bicis disponibles o 0 docks libres (entre 0 y 1).
- **No se descarta ninguna fila.** Se cuentan las fotos con problemas (estación que no alquila, capacidad 0) para que luego en el tercer notebook decidamos qué hacer con ellas.

In [10]:
inicio = time.time()
con.execute(f"""
COPY (
    SELECT TRY_CAST(station_id AS INTEGER)                                     AS station_id,
           date_trunc('hour', committed_at_utc AT TIME ZONE '{TZ}')            AS hora_local,
           count(*)                                                            AS n_fotos,
           avg(num_bikes_available)                                            AS bicis_disponibles,
           avg(num_bikes_disabled)                                             AS bicis_rotas,
           avg(num_docks_available)                                            AS anclajes_libres,
           avg(num_docks_disabled)                                             AS anclajes_rotos,
           avg(capacity)                                                       AS capacidad_publicada,
           avg(num_bikes_available + num_docks_available)                      AS capacidad_util,
           avg(CASE WHEN num_bikes_available = 0 THEN 1 ELSE 0 END)            AS prop_vacia,
           avg(CASE WHEN num_docks_available = 0 THEN 1 ELSE 0 END)            AS prop_llena,
           sum(CASE WHEN NOT is_renting THEN 1 ELSE 0 END)                     AS n_fotos_sin_alquiler,
           sum(CASE WHEN capacity = 0 THEN 1 ELSE 0 END)                       AS n_fotos_capacidad_cero
    FROM fotos
    GROUP BY 1, 2
    ORDER BY 1, 2
) TO '{(INTERIM_DIR / 'fotos_hora.parquet').as_posix()}' (FORMAT parquet)
""")
print(f"fotos_hora.parquet generado en {time.time() - inicio:.0f} s")

fotos_hora.parquet generado en 1 s


In [11]:
con.execute(f"CREATE OR REPLACE VIEW fotos_hora AS SELECT * FROM read_parquet('{(INTERIM_DIR / 'fotos_hora.parquet').as_posix()}')")
consulta("""
SELECT count(*)                   AS filas_estacion_hora,
       count(DISTINCT station_id) AS estaciones,
       count(DISTINCT hora_local) AS horas_con_fotos,
       min(hora_local)            AS desde,
       max(hora_local)            AS hasta
FROM fotos_hora
""")

,filas_estacion_hora,estaciones,horas_con_fotos,desde,hasta
0,6708782,414,17115,2024-04-10 18:00:00,2026-09-27 11:00:00


In [12]:
consulta("SELECT * FROM fotos_hora USING SAMPLE 5")

,station_id,hora_local,n_fotos,bicis_disponibles,bicis_rotas,anclajes_libres,anclajes_rotos,capacidad_publicada,capacidad_util,prop_vacia,prop_llena,n_fotos_sin_alquiler,n_fotos_capacidad_cero
0,2,2024-11-29 03:00:00,4,19.0,1.0,20.0,0.0,40.0,39.0,0.0,0.0,0.0,0.0
1,2,2025-03-07 02:00:00,4,8.0,0.0,32.0,0.0,40.0,40.0,0.0,0.0,0.0,0.0
2,9,2025-03-31 03:00:00,4,1.5,0.0,22.5,0.0,24.0,24.0,0.0,0.0,0.0,0.0
3,98,2025-01-19 01:00:00,4,2.0,6.0,8.0,0.0,16.0,10.0,0.0,0.0,0.0,0.0
4,98,2025-05-31 12:00:00,4,1.0,4.0,11.0,0.0,16.0,12.0,0.0,0.0,0.0,0.0


## 3. Viajes realizados (BA Data)

**Qué son.** Un registro por viaje en Ecobici: estación y hora de origen, estación y hora de destino, duración, usuario, modelo de bici y género declarado. Es la **demanda real**: la cantidad de retiros por estación y hora es el target del proyecto.

**Formato.** Un CSV por año (2024, 2025 y 2026), de 500 a 830 MB cada uno. Las fechas vienen en hora local sin zona horaria.

**Variables:**

| Variable | Tipo | Significado |
|---|---|---|
| `id_recorrido` | entero | Identificador del viaje |
| `duracion_recorrido` | entero (segundos) | Duración del viaje |
| `fecha_origen_recorrido`, `fecha_destino_recorrido` | fecha y hora | Inicio y fin del viaje (hora local) |
| `id_estacion_origen`, `id_estacion_destino` | entero | Estaciones de origen y destino (mismo identificador que `station_id` de las fotos) |
| `nombre_estacion_*`, `direccion_estacion_*` | texto | Nombre y dirección de cada estación |
| `long_estacion_*`, `lat_estacion_*` | decimal | Ubicación de cada estación |
| `id_usuario` | entero | Identificador del usuario |
| `modelo_bicicleta` | texto | Modelo de bici (`FIT`, `ICONIC`) |
| `genero` | texto | Género declarado (`MALE`, `FEMALE`, `OTHER`) |

In [13]:
archivos_viajes = {anio: next((BA_DATA_DIR / f"recorridos-realizados-{anio}").glob("*.csv")) for anio in ANIOS}
for anio, ruta in archivos_viajes.items():
    print(anio, ruta.relative_to(DATA_DIR).as_posix(), f"({ruta.stat().st_size / 1e6:.0f} MB)")

2024 ba-ecobici-datasets/recorridos-realizados-2024/badata_ecobici_recorridos_realizados_2024.csv (802 MB)
2025 ba-ecobici-datasets/recorridos-realizados-2025/Trips.csv (832 MB)
2026 ba-ecobici-datasets/recorridos-realizados-2026/Trips.csv (522 MB)


### 3.1 Los tres años no tienen el mismo esquema

Antes de unir los archivos se comparan sus encabezados.

In [14]:
encabezados = {}
for anio, ruta in archivos_viajes.items():
    with open(ruta, encoding="utf-8") as archivo:
        encabezados[anio] = archivo.readline().rstrip("\r\n").split(",")

comparacion = pd.DataFrame(encabezados)
comparacion["iguales"] = comparacion[ANIOS].nunique(axis=1) == 1
comparacion

,2024,2025,2026,iguales
0,﻿id_recorrido,Id_recorrido,Id_recorrido,False
1,duracion_recorrido,duracion_recorrido,duracion_recorrido,True
2,fecha_origen_recorrido,fecha_origen_recorrido,fecha_origen_recorrido,True
3,id_estacion_origen,id_estacion_origen,id_estacion_origen,True
4,nombre_estacion_origen,nombre_estacion_origen,nombre_estacion_origen,True
5,direccion_estacion_origen,direccion_estacion_origen,direccion_estacion_origen,True
6,long_estacion_origen,long_estacion_origen,long_estacion_origen,True
7,lat_estacion_origen,lat_estacion_origen,lat_estacion_origen,True
8,fecha_destino_recorrido,fecha_destino_recorrido,fecha_destino_recorrido,True
9,id_estacion_destino,id_estacion_destino,id_estacion_destino,True


In [15]:
print("Primera columna de cada año, tal como está en el archivo:")
for anio in ANIOS:
    print(f"  {anio}: {encabezados[anio][0]!r}")

Primera columna de cada año, tal como está en el archivo:
  2024: '\ufeffid_recorrido'
  2025: 'Id_recorrido'
  2026: 'Id_recorrido'


Además de los nombres, en las primeras filas se ve que cambia el **formato de los valores**:

| | 2024 | 2025 y 2026 |
|---|---|---|
| Nombre de la primera columna | `id_recorrido`, con el carácter invisible `\ufeff` (BOM) delante | `Id_recorrido` |
| Columna de género | `genero` | `género` |
| IDs de viaje, estación y usuario | `513`, `992557.0` (usuario como decimal) | `513BAEcobici` |
| Duración | número | texto con separador de miles (`"1,164"`) |

Por eso la carga se hace con una función que unifica el esquema.

In [16]:
# Primeras 3 filas de cada año, leídas como texto: id de viaje, duración, estación de origen, usuario y género
for anio, ruta in archivos_viajes.items():
    print(anio)
    display(pd.read_csv(ruta, nrows=3, dtype=str).iloc[:, [0, 1, 3, 14, 16]])

2024


,id_recorrido,duracion_recorrido,id_estacion_origen,id_usuario,genero
0,20428222,568,513,992557.0,MALE
1,20431744,1355,460,320782.0,FEMALE
2,20429936,0,467,828678.0,FEMALE


2025


,Id_recorrido,duracion_recorrido,id_estacion_origen,id_usuario,género
0,27699090BAEcobici,800,146BAEcobici,1177390BAEcobici,FEMALE
1,24622664BAEcobici,571,70BAEcobici,357363BAEcobici,OTHER
2,24621373BAEcobici,348,135BAEcobici,693254BAEcobici,MALE


2026


,Id_recorrido,duracion_recorrido,id_estacion_origen,id_usuario,género
0,30516368BAEcobici,"1,178",555BAEcobici,985478BAEcobici,MALE
1,30266718BAEcobici,202,188BAEcobici,1182509BAEcobici,MALE
2,30262110BAEcobici,439,202BAEcobici,1106930BAEcobici,MALE


### 3.2 Carga con esquema unificado

La función `cargar_viajes(anio)` arma la consulta que lee el CSV de un año y devuelve las columnas con el mismo nombre y tipo para los tres años:

1. **Nombres por posición.** Los tres archivos tienen las mismas 17 columnas en el mismo orden. Se ignoran los encabezados originales y se asignan nombres únicos (`names=`). Así se resuelven de una vez `Id_recorrido`/`id_recorrido`, `género`/`genero` y el BOM.
2. **Se lee todo como texto** (`all_varchar`) para que ningún valor raro haga fallar la lectura. Los tipos se asignan después.
3. **IDs:** se quita el sufijo `BAEcobici` y se convierten a entero. El paso intermedio por decimal resuelve los IDs de usuario de 2024 (`992557.0`).
4. **Duración:** se quita la coma de los miles y se convierte a número.
5. **Fechas y coordenadas:** se convierten a fecha y hora y a decimal.
6. **`TRY_CAST`:** si un valor no se puede convertir, queda como nulo en lugar de cortar la carga. Los nulos se cuentan más abajo.
7. **`anio_archivo`:** se agrega el año del archivo de origen, para poder comparar años en el diagnóstico.

No se filtra ninguna fila ni se corrigen valores (espacios sobrantes, outliers): eso es parte de la limpieza del notebook centrado en transformar y limpiar los datos.

In [17]:
COLUMNAS_VIAJES = [
    "id_recorrido", "duracion_recorrido", "fecha_origen_recorrido",
    "id_estacion_origen", "nombre_estacion_origen", "direccion_estacion_origen",
    "long_estacion_origen", "lat_estacion_origen", "fecha_destino_recorrido",
    "id_estacion_destino", "nombre_estacion_destino", "direccion_estacion_destino",
    "long_estacion_destino", "lat_estacion_destino", "id_usuario",
    "modelo_bicicleta", "genero",
]


def a_entero(columna):
    """Quita el sufijo 'BAEcobici' y convierte a entero (pasando por decimal por los '992557.0' de 2024)."""
    return f"TRY_CAST(TRY_CAST(replace({columna}, 'BAEcobici', '') AS DOUBLE) AS BIGINT)"


def cargar_viajes(anio):
    """Devuelve la consulta SQL que lee los viajes de un año con el esquema unificado."""
    ruta = archivos_viajes[anio].as_posix()
    return f"""
    SELECT {anio}                                                   AS anio_archivo,
           {a_entero('id_recorrido')}                               AS id_recorrido,
           TRY_CAST(replace(duracion_recorrido, ',', '') AS DOUBLE) AS duracion_seg,
           TRY_CAST(fecha_origen_recorrido AS TIMESTAMP)            AS fecha_origen,
           {a_entero('id_estacion_origen')}                         AS id_estacion_origen,
           nombre_estacion_origen,
           direccion_estacion_origen,
           TRY_CAST(long_estacion_origen AS DOUBLE)                 AS long_estacion_origen,
           TRY_CAST(lat_estacion_origen AS DOUBLE)                  AS lat_estacion_origen,
           TRY_CAST(fecha_destino_recorrido AS TIMESTAMP)           AS fecha_destino,
           {a_entero('id_estacion_destino')}                        AS id_estacion_destino,
           nombre_estacion_destino,
           direccion_estacion_destino,
           TRY_CAST(long_estacion_destino AS DOUBLE)                AS long_estacion_destino,
           TRY_CAST(lat_estacion_destino AS DOUBLE)                 AS lat_estacion_destino,
           {a_entero('id_usuario')}                                 AS id_usuario,
           modelo_bicicleta,
           genero
    FROM read_csv('{ruta}', header = true, all_varchar = true, names = {COLUMNAS_VIAJES})
    """


print(cargar_viajes(2025))


    SELECT 2025                                                   AS anio_archivo,
           TRY_CAST(TRY_CAST(replace(id_recorrido, 'BAEcobici', '') AS DOUBLE) AS BIGINT)                               AS id_recorrido,
           TRY_CAST(replace(duracion_recorrido, ',', '') AS DOUBLE) AS duracion_seg,
           TRY_CAST(fecha_origen_recorrido AS TIMESTAMP)            AS fecha_origen,
           TRY_CAST(TRY_CAST(replace(id_estacion_origen, 'BAEcobici', '') AS DOUBLE) AS BIGINT)                         AS id_estacion_origen,
           nombre_estacion_origen,
           direccion_estacion_origen,
           TRY_CAST(long_estacion_origen AS DOUBLE)                 AS long_estacion_origen,
           TRY_CAST(lat_estacion_origen AS DOUBLE)                  AS lat_estacion_origen,
           TRY_CAST(fecha_destino_recorrido AS TIMESTAMP)           AS fecha_destino,
           TRY_CAST(TRY_CAST(replace(id_estacion_destino, 'BAEcobici', '') AS DOUBLE) AS BIGINT)                        AS

Se unen los tres años con un UNION ALL y se guardan en un solo parquet.

In [18]:
inicio = time.time()
union_viajes = " UNION ALL ".join(cargar_viajes(anio) for anio in ANIOS)
con.execute(f"COPY ({union_viajes}) TO '{(INTERIM_DIR / 'viajes.parquet').as_posix()}' (FORMAT parquet)")
print(f"viajes.parquet generado en {time.time() - inicio:.0f} s")

con.execute(f"CREATE OR REPLACE VIEW viajes AS SELECT * FROM read_parquet('{(INTERIM_DIR / 'viajes.parquet').as_posix()}')")
consulta("DESCRIBE viajes")[["column_name", "column_type"]]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

viajes.parquet generado en 4 s


,column_name,column_type
0,anio_archivo,INTEGER
1,id_recorrido,BIGINT
2,duracion_seg,DOUBLE
3,fecha_origen,TIMESTAMP
4,id_estacion_origen,BIGINT
5,nombre_estacion_origen,VARCHAR
6,direccion_estacion_origen,VARCHAR
7,long_estacion_origen,DOUBLE
8,lat_estacion_origen,DOUBLE
9,fecha_destino,TIMESTAMP


In [19]:
consulta("""
SELECT anio_archivo,
       count(*)                           AS viajes,
       min(fecha_origen)                  AS primer_viaje,
       max(fecha_origen)                  AS ultimo_viaje,
       count(DISTINCT id_estacion_origen) AS estaciones_origen
FROM viajes
GROUP BY 1
ORDER BY 1
""")

,anio_archivo,viajes,primer_viaje,ultimo_viaje,estaciones_origen
0,2024,3559284,2024-01-01 00:06:50,2024-12-31 23:58:25,395
1,2025,3156182,2025-01-01 00:01:40,2025-12-31 23:58:28,403
2,2026,1979625,2026-01-01 00:03:13,2026-08-31 23:59:54,405


**Nulos por columna y año.** Un nulo puede ser un campo vacío en el CSV o un valor que `TRY_CAST` no pudo convertir. El segundo notebook centrado en calidad va a separar ambos casos.

In [20]:
columnas = [c for c in consulta("DESCRIBE viajes")["column_name"] if c != "anio_archivo"]
conteos = ", ".join(f'count(*) - count("{c}") AS "{c}"' for c in columnas)
nulos_viajes = consulta(f"SELECT anio_archivo, {conteos} FROM viajes GROUP BY 1 ORDER BY 1").set_index("anio_archivo").T
nulos_viajes

anio_archivo,2024,2025,2026
id_recorrido,0,0,0
duracion_seg,0,0,0
fecha_origen,0,0,0
id_estacion_origen,0,0,0
nombre_estacion_origen,0,18,4
direccion_estacion_origen,0,0,0
long_estacion_origen,0,18,4
lat_estacion_origen,0,18,4
fecha_destino,3379,0,0
id_estacion_destino,0,1,1


Viajes por mes. Sirve para confirmar la cobertura de cada archivo, en particular el último mes de 2026.

In [21]:
viajes_por_mes = consulta("""
SELECT strftime(fecha_origen, '%Y-%m') AS mes, count(*) AS viajes
FROM viajes
GROUP BY 1
ORDER BY 1
""")
viajes_por_mes

,mes,viajes
0,2024-01,293151
1,2024-02,276602
2,2024-03,313501
3,2024-04,281259
4,2024-05,266596
5,2024-06,240287
6,2024-07,238056
7,2024-08,246193
8,2024-09,347735
9,2024-10,373398


In [22]:
muestra_viajes = consulta("SELECT * FROM viajes USING SAMPLE 100000")
muestra_viajes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 18 columns):
 #   Column                      Non-Null Count   Dtype         
---  ------                      --------------   -----         
 0   anio_archivo                100000 non-null  int32         
 1   id_recorrido                100000 non-null  int64         
 2   duracion_seg                100000 non-null  float64       
 3   fecha_origen                100000 non-null  datetime64[us]
 4   id_estacion_origen          100000 non-null  int64         
 5   nombre_estacion_origen      100000 non-null  object        
 6   direccion_estacion_origen   100000 non-null  object        
 7   long_estacion_origen        100000 non-null  float64       
 8   lat_estacion_origen         100000 non-null  float64       
 9   fecha_destino               99965 non-null   datetime64[us]
 10  id_estacion_destino         100000 non-null  int64         
 11  nombre_estacion_destino     100000 non-n

In [23]:
muestra_viajes.head()

,anio_archivo,id_recorrido,duracion_seg,fecha_origen,id_estacion_origen,nombre_estacion_origen,direccion_estacion_origen,long_estacion_origen,lat_estacion_origen,fecha_destino,id_estacion_destino,nombre_estacion_destino,direccion_estacion_destino,long_estacion_destino,lat_estacion_destino,id_usuario,modelo_bicicleta,genero
0,2024,21287291,622.0,2024-04-05 21:32:08,372,322 - MARTÍ Y DIRECTORIO,"404 Marti, Jose",-58.470136,-34.636406,2024-04-05 21:42:30,407,330 - Parque Avellaneda,Remedios 3895,-58.478266,-34.642425,1119522,FIT,FEMALE
1,2025,24597681,2175.0,2025-01-08 17:03:41,82,082 - HOSPITAL ITALIANO,394 Rawson & Potosi,-58.426395,-34.607892,2025-01-08 17:39:56,259,290 - Club Excursionistas,Migueletes 1782 (y La Pampa),-58.444176,-34.559152,713849,FIT,MALE
2,2025,26590421,999.0,2025-07-02 08:42:28,277,292 - PLAZA BOLIVIA,Olleros Av. & Del Libertador Av.,-58.436115,-34.563539,2025-07-02 08:59:07,234,344 - UNIVERSIDAD DI TELLA,Figueroa Alcorta 7285,-58.446802,-34.547232,28510,ICONIC,FEMALE
3,2025,24911384,518.0,2025-02-01 17:03:39,124,124 - UGARTECHE,Juan María Gutiérrez 2069,-58.412625,-34.581114,2025-02-01 17:12:17,517,332 - PACÍFICO II,Av. Int. Bullrich 300,-58.424814,-34.574501,85567,ICONIC,MALE
4,2024,21643361,1424.0,2024-05-08 14:27:06,65,065 - Julián Álvarez,3822 Guemes,-58.415787,-34.587312,2024-05-08 14:50:50,87,087 - Guayaquil,Guayaquil & Doblas,-58.431494,-34.619845,1132093,ICONIC,MALE


## 4. Clima observado (API de Open-Meteo)

**Por qué esta fuente.** El repositorio de las fotos también guarda pronósticos de clima, pero tienen dos problemas: cada hora tiene hasta 37 versiones distintas del pronóstico y hay huecos en los mismos meses que las fotos (sección 4.1). Open-Meteo ofrece el **clima observado** por hora, completo y en un solo pedido a su API.

**Parámetros del pedido:**

- **Ubicación:** el Obelisco (-34,6037; -58,3816), como punto representativo de la ciudad.
- **Período:** del 01/01/2024 al 20/09/2026. El archivo histórico se publica con unos días de demora, así que no se piden los últimos días.
- **Variables por hora:** temperatura (°C), precipitación (mm) y velocidad del viento (km/h).
- **Zona horaria:** Buenos Aires, igual que los viajes.

La API es gratuita y no pide clave. Sus datos se publican bajo licencia CC BY 4.0, que nos obliga a citar la fuente (ver Referencias).

In [24]:
URL_OPEN_METEO = "https://archive-api.open-meteo.com/v1/archive"
parametros = {
    "latitude": -34.6037,
    "longitude": -58.3816,
    "start_date": "2024-01-01",
    "end_date": "2026-09-20",
    "hourly": "temperature_2m,precipitation,wind_speed_10m",
    "timezone": TZ,
}

respuesta = requests.get(URL_OPEN_METEO, params=parametros, timeout=120)
print("Código de estado:", respuesta.status_code)
respuesta.raise_for_status()

datos_clima = respuesta.json()
print("Unidades:", datos_clima["hourly_units"])

Código de estado: 200
Unidades: {'time': 'iso8601', 'temperature_2m': '°C', 'precipitation': 'mm', 'wind_speed_10m': 'km/h'}


In [25]:
clima = pd.DataFrame(datos_clima["hourly"]).rename(columns={
    "time": "hora_local",
    "temperature_2m": "temperatura_c",
    "precipitation": "precipitacion_mm",
    "wind_speed_10m": "viento_kmh",
})
clima["hora_local"] = pd.to_datetime(clima["hora_local"])
clima.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23856 entries, 0 to 23855
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   hora_local        23856 non-null  datetime64[ns]
 1   temperatura_c     23856 non-null  float64       
 2   precipitacion_mm  23856 non-null  float64       
 3   viento_kmh        23856 non-null  float64       
dtypes: datetime64[ns](1), float64(3)
memory usage: 745.6 KB


In [26]:
clima.describe()

,hora_local,temperatura_c,precipitacion_mm,viento_kmh
count,23856,23856.000000,23856.000000,23856.000000
mean,2025-05-11 23:30:00,17.679791,0.122527,12.386603
min,2024-01-01 00:00:00,-1.400000,0.000000,0.000000
25%,2024-09-05 11:45:00,12.400000,0.000000,8.300000
50%,2025-05-11 23:30:00,17.800000,0.000000,11.600000
75%,2026-01-15 11:15:00,22.800000,0.000000,15.600000
max,2026-09-20 23:00:00,38.700000,21.800000,41.400000
std,NaN,6.676522,0.752982,5.794593


In [27]:
print("Nulos por columna:")
print(clima.isnull().sum())

clima.to_parquet(INTERIM_DIR / "clima.parquet", index=False)
print("\nclima.parquet guardado:", clima.shape)

Nulos por columna:
hora_local          0
temperatura_c       0
precipitacion_mm    0
viento_kmh          0
dtype: int64

clima.parquet guardado: (23856, 4)


### 4.1 Pronósticos de clima (sólo como referencia)

Se describen para justificar por qué no se usan como fuente principal. En producción, en cambio, sólo va a haber pronóstico: eso quedará para los "Siguientes pasos".

| Variable | Significado |
|---|---|
| `forecast_at_utc` | Hora pronosticada (UTC) |
| `temperature`, `rain`, `wind_speed` | Temperatura (°C), lluvia (mm) y viento (km/h) pronosticados |
| `committed_at_utc` | Momento en que se publicó el pronóstico |

In [28]:
con.execute(f"CREATE OR REPLACE VIEW pronosticos AS SELECT * FROM read_parquet('{PRONOSTICOS_DIR.as_posix()}/*/*.parquet')")
consulta("""
WITH versiones AS (
    SELECT forecast_at_utc, count(*) AS n_versiones
    FROM pronosticos
    GROUP BY 1
)
SELECT count(*)                  AS horas_pronosticadas,
       min(forecast_at_utc)      AS desde,
       max(forecast_at_utc)      AS hasta,
       median(n_versiones)       AS versiones_por_hora_mediana,
       max(n_versiones)          AS versiones_por_hora_max
FROM versiones
""")

,horas_pronosticadas,desde,hasta,versiones_por_hora_mediana,versiones_por_hora_max
0,18216,2024-04-09 21:00:00-03:00,2026-09-27 20:00:00-03:00,6.0,37


## 5. Feriados (API de ArgentinaDatos)

Se pide la lista de feriados nacionales de cada año a `api.argentinadatos.com/v1/feriados/{año}`. Cada feriado trae la fecha, el tipo y el nombre. Los tipos son:

- `inamovible`: siempre cae en la misma fecha.
- `trasladable`: el gobierno puede moverlo a un lunes o un viernes.
- `puente`: puente turístico no laborable.

**Limitación conocida:** la API no incluye los **días no laborables**, como Jueves Santo (28/03/2024 y 17/04/2025). Por ahora se usa la API tal cual; queda como pendiente.

El resultado se guarda en `data/feriados.csv`, que también se versiona en el repositorio (es chico) por si la API no responde.

In [29]:
feriados_por_anio = []
for anio in ANIOS:
    respuesta = requests.get(f"https://api.argentinadatos.com/v1/feriados/{anio}", timeout=30)
    print(anio, "-> código de estado", respuesta.status_code)
    respuesta.raise_for_status()
    feriados_por_anio.append(pd.DataFrame(respuesta.json()))

feriados = pd.concat(feriados_por_anio, ignore_index=True)
feriados["fecha"] = pd.to_datetime(feriados["fecha"])
feriados.to_csv(REPO_DIR / "data" / "feriados.csv", index=False)
feriados.head(10)

2024 -> código de estado 200


2025 -> código de estado 200


2026 -> código de estado 200


,fecha,tipo,nombre
0,2024-01-01,inamovible,Año nuevo
1,2024-02-12,inamovible,Carnaval
2,2024-02-13,inamovible,Carnaval
3,2024-03-24,inamovible,Día Nacional de la Memoria por la Verdad y la ...
4,2024-03-29,inamovible,Viernes Santo
5,2024-04-01,puente,Feriado puente turístico
6,2024-04-02,inamovible,Día del Veterano y de los Caídos en la Guerra ...
7,2024-05-01,inamovible,Día del Trabajador
8,2024-05-25,inamovible,Día de la Revolución de Mayo
9,2024-06-17,trasladable,Paso a la Inmortalidad del General Martín Güemes


In [30]:
pd.crosstab(feriados["fecha"].dt.year, feriados["tipo"], margins=True)

tipo,inamovible,puente,trasladable,All
fecha,,,,
2024,12,3,4,19
2025,12,4,4,20
2026,12,3,4,19
All,36,10,12,58


## 6. Datos complementarios

### 6.1 Ciclovías

Un tramo de ciclovía por fila, con su geometría en formato WKT (`LINESTRING`), el tipo, la comuna, el barrio y la longitud en metros. Podría servir para una variable espacial por estación (por ejemplo, metros de ciclovía cerca). Por ahora sólo se describe; el archivo es chico y se lee directamente del crudo cuando haga falta.

In [31]:
ciclovias = pd.read_csv(BA_DATA_DIR / "ciclovias.csv", encoding="utf-8-sig")
ciclovias.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2764 entries, 0 to 2763
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   id          2764 non-null   int64  
 1   nombre      2764 non-null   object 
 2   tipo        2764 non-null   object 
 3   comuna      2764 non-null   int64  
 4   barrio      2764 non-null   object 
 5   longitud_m  2764 non-null   float64
 6   geometry    2764 non-null   object 
dtypes: float64(1), int64(2), object(4)
memory usage: 151.3+ KB


In [32]:
ciclovias.head()

,id,nombre,tipo,comuna,barrio,longitud_m,geometry
0,10650,"PERON, JUAN DOMINGO, TTE. GENERAL",Ciclovías mano única,1,San Nicolas,52.98,LINESTRING (-58.3802625790606 -34.606056085163...
1,14588,RINCON,Ciclovías,3,San Cristobal,118.72,LINESTRING (-58.395928300929576 -34.6195618346...
2,7180,NUEVA YORK,Ciclovías,15,Agronomia,129.98,LINESTRING (-58.49641933160354 -34.59071404109...
3,15268,ESTADOS UNIDOS,Ciclovías,5,Boedo,71.00,LINESTRING (-58.42270933136431 -34.62306774122...
4,17386,ESTRADA,Ciclovías,7,Parque Chacabuco,132.34,LINESTRING (-58.42598142884307 -34.63045069879...


In [33]:
ciclovias["tipo"].value_counts()

tipo
Ciclovías               2469
Ciclovías mano única     295
Name: count, dtype: int64

## 7. Archivos generados

In [34]:
salidas = sorted(INTERIM_DIR.glob("*.parquet")) + [REPO_DIR / "data" / "feriados.csv"]
resumen = []
for ruta in salidas:
    filas = consulta(f"SELECT count(*) AS n FROM '{ruta.as_posix()}'")["n"].iloc[0]
    resumen.append({
        "archivo": ruta.relative_to(REPO_DIR).as_posix(),
        "filas": filas,
        "MB": round(ruta.stat().st_size / 1e6, 1),
    })
pd.DataFrame(resumen)

,archivo,filas,MB
0,data/interim/clima.parquet,23856,0.3
1,data/interim/fotos_hora.parquet,6708782,37.3
2,data/interim/viajes.parquet,8695091,320.9
3,data/feriados.csv,58,0.0


## 8. Diccionario de datos de los archivos generados

**`data/interim/fotos_hora.parquet`**: una fila por estación y hora con al menos una foto.

| Variable | Tipo | Significado |
|---|---|---|
| `station_id` | entero | Estación |
| `hora_local` | fecha y hora | Hora (Buenos Aires), truncada |
| `n_fotos` | entero | Fotos de esa estación en esa hora |
| `bicis_disponibles` | decimal | Promedio de bicis disponibles |
| `bicis_rotas` | decimal | Promedio de bicis rotas o bloqueadas |
| `anclajes_libres` | decimal | Promedio de anclajes libres |
| `anclajes_rotos` | decimal | Promedio de anclajes fuera de servicio |
| `capacidad_publicada` | decimal | Promedio de `capacity` |
| `capacidad_util` | decimal | Promedio de bicis disponibles + anclajes libres |
| `prop_vacia` | decimal (0–1) | Fracción de fotos con 0 bicis disponibles |
| `prop_llena` | decimal (0–1) | Fracción de fotos con 0 anclajes libres |
| `n_fotos_sin_alquiler` | entero | Fotos con `is_renting = False` |
| `n_fotos_capacidad_cero` | entero | Fotos con `capacity = 0` |

**`data/interim/viajes.parquet`**: un viaje por fila, 2024–2026, con el esquema unificado.

| Variable | Tipo | Significado |
|---|---|---|
| `anio_archivo` | entero | Año del archivo de origen |
| `id_recorrido` | entero | Identificador del viaje |
| `duracion_seg` | decimal | Duración en segundos |
| `fecha_origen`, `fecha_destino` | fecha y hora | Inicio y fin (hora local) |
| `id_estacion_origen`, `id_estacion_destino` | entero | Estaciones (cruzan con `station_id`) |
| `nombre_estacion_*`, `direccion_estacion_*` | texto | Sin limpiar (espacios sobrantes incluidos) |
| `long_estacion_*`, `lat_estacion_*` | decimal | Ubicación de las estaciones |
| `id_usuario` | entero | Usuario |
| `modelo_bicicleta` | texto | `FIT` o `ICONIC` |
| `genero` | texto | Género declarado |

**`data/interim/clima.parquet`**: una fila por hora.

| Variable | Tipo | Significado |
|---|---|---|
| `hora_local` | fecha y hora | Hora (Buenos Aires) |
| `temperatura_c` | decimal | Temperatura a 2 m (°C) |
| `precipitacion_mm` | decimal | Precipitación en la hora (mm) |
| `viento_kmh` | decimal | Velocidad del viento a 10 m (km/h) |

**`data/feriados.csv`**: un feriado por fila.

| Variable | Tipo | Significado |
|---|---|---|
| `fecha` | fecha | Día del feriado |
| `tipo` | texto | `inamovible`, `trasladable` o `puente` |
| `nombre` | texto | Motivo del feriado |


## 9. Justificación de la elección de los datos

- **Los viajes miden la demanda real.** Cada retiro de una bici queda registrado con su estación y su hora, que es exactamente lo que el proyecto quiere predecir. Cubren desde enero de 2024 hasta 2026, con más de dos años de historia.
- **Las fotos de las distintas estaciones en base al feedback recibido previamente.** Son la única fuente con las bicis rotas y los docks fuera de servicio de cada estación. Permiten calcular la capacidad útil y detectar cuándo una estación "vacía" o "llena" lo está por bicis que no se pueden usar. También muestran cuándo una estación no tenía bicis para ofrecer, algo que los viajes no pueden mostrar: si no hubo retiros, no se sabe si fue por falta de demanda o de bicis.
- **Las dos fuentes se pueden cruzar.** El identificador de estación de los viajes coincide con el `station_id` de las fotos.
- **El clima y los feriados responden a la restricción de la propuesta.** En la propuesta, destacamos como riesgo la "falta de variables externas como factores climáticos". Se incorporan por API, con datos observados y completos.
- **Todas las fuentes son abiertas y citables**, y se pueden volver a descargar para actualizar el análisis.

## 10. Referencias

- Halford, M. *bike-sharing-history* [dataset]. GitHub. https://github.com/MaxHalford/bike-sharing-history (licencia MIT; consultado el 19/09/2026).
- Gobierno de la Ciudad de Buenos Aires. BA Data, "Bicicletas públicas": recorridos realizados 2024, 2025 y 2026, y usuarios. https://data.buenosaires.gob.ar (descargado el 19/09/2026).
- Gobierno de la Ciudad de Buenos Aires. BA Data, "Ciclovías". https://data.buenosaires.gob.ar (descargado el 19/09/2026).
- Open-Meteo. Historical Weather API. https://open-meteo.com (licencia CC BY 4.0).
- ArgentinaDatos. API de feriados. https://api.argentinadatos.com